# LoopBike: notebook starter

The data half of the starter kit. It loads LoopBike's six files, gives you a quick way to profile them, and draws one rough chart. Everything after that is yours.

**Before you start:** `pip install -r requirements.txt`, then open this notebook in Jupyter, VS Code or Google Colab.

Nothing here has been cleaned. Your Tier 1 job is to find out what's wrong, decide what to do about it, and record each decision in `DATA_QUALITY_LOG.md`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 20)

# The notebook lives in notebook/, the data in data/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATA = ROOT / "data"
TIER3 = ROOT / "tier3"
print("Reading data from", DATA.resolve())

## Load everything as text

`dtype=str` stops pandas quietly guessing types. You decide how each column should be converted, and you'll see anything that doesn't fit.

In [ ]:
stations = pd.read_csv(DATA / "stations.csv", dtype=str)
trips = pd.read_csv(DATA / "trips.csv", dtype=str)
members = pd.read_csv(DATA / "members.csv", dtype=str)
bikes = pd.read_csv(DATA / "bikes.csv", dtype=str)
weather = pd.read_csv(DATA / "weather.csv", dtype=str)
maintenance = pd.read_csv(DATA / "maintenance.csv", dtype=str)

for name, df in [("stations", stations), ("trips", trips), ("members", members),
                 ("bikes", bikes), ("weather", weather), ("maintenance", maintenance)]:
    print(f"{name:12} {len(df):>7,} rows  {df.shape[1]} columns")

In [ ]:
trips.head(10)

## Profile a table

A quick summary of every column: how many values are missing, how many distinct values there are, and the most common ones. Use it on each table and ask: *is that what I'd expect?*

In [ ]:
def profile(df, top=5):
    rows = []
    for col in df.columns:
        s = df[col]
        rows.append({
            "column": col,
            "missing": int(s.isna().sum()),
            "distinct": int(s.nunique()),
            "most common": ", ".join(f"{v!r} ({n})" for v, n in s.value_counts().head(top).items()),
        })
    return pd.DataFrame(rows).set_index("column")


profile(trips)

In [ ]:
profile(members)

## A first chart

A rough first attempt at trips per day. It only understands one timestamp format, so check the printout before trusting the chart.

In [ ]:
starts = pd.to_datetime(trips["start_time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
print(f"{starts.isna().sum():,} of {len(starts):,} start times did not parse with this format")

starts.dt.date.value_counts().sort_index().plot(figsize=(12, 3), title="Trips per day (first attempt)")
plt.show()

## Your cleaning goes here

Keep it as code you can re-run from the top rather than one-off fixes. If more data turns up later, you'll want to run the same steps again.

When you have clean tables, save them to `data/clean/` in the shape agreed in `DATA_CONTRACT.md`, so your team's tool can read them.

In [ ]:
CLEAN = DATA / "clean"
CLEAN.mkdir(exist_ok=True)

# trips_clean = ...
# trips_clean.to_csv(CLEAN / "trips.csv", index=False)

## Tier 3 starting point: a naive baseline

Option A asks you to predict **customer departures per station per day** for 21 to 27 September (see `tier3/README.md`). The simplest possible prediction is "the same as a typical recent day of the same weekday". Your model has to beat that.

This function needs **clean** trips with two columns: `station_id` (like `LB007`) and `date`. It won't work until you've got those.

In [ ]:
def weekday_baseline(clean_trips, last_n_days=28):
    """Average departures per station for each weekday over the last `last_n_days` of data."""
    t = clean_trips[["station_id", "date"]].copy()
    t["date"] = pd.to_datetime(t["date"]).dt.normalize()
    end = t["date"].max()
    days = pd.date_range(end - pd.Timedelta(days=last_n_days - 1), end)
    template = pd.read_csv(TIER3 / "demand_predictions_template.csv")

    daily = (
        t[t["date"].isin(days)]
        .groupby(["station_id", "date"]).size()
        .reindex(pd.MultiIndex.from_product([template["station_id"].unique(), days],
                                            names=["station_id", "date"]), fill_value=0)
        .rename("departures").reset_index()
    )
    typical = daily.groupby([daily["station_id"], daily["date"].dt.weekday])["departures"].mean()

    weekdays = pd.to_datetime(template["date"]).dt.weekday
    template["predicted_departures"] = [
        round(typical.get((s, w), 0.0), 2) for s, w in zip(template["station_id"], weekdays)
    ]
    return template


# predictions = weekday_baseline(trips_clean)
# predictions.to_csv(TIER3 / "demand_predictions.csv", index=False)